# 03 - Regression & Modeling Analyst: Korelasi & Regresi Linear Berganda  (Tahap 4)

**Peran:** Regression & Modeling Analyst - *korelasi, model regresi & uji asumsi residual*
**Dikerjakan oleh:** _(isi nama)_

### Cerita singkat tugasmu
Kamu membangun **model prediksi sederhana** (inilah "model"-nya proyek ini - bukan deep learning, tapi prinsip dasar yang sama: belajar pola dari data).
Pertanyaan riset dari dosen: **"Bagaimana pengaruh Volume Kendaraan dan Kecepatan Angin terhadap Indeks PM2.5?"**

$$\widehat{PM2.5} = b_0 + b_1 \cdot Volume + b_2 \cdot Angin$$

| Bagian | Isi | Dari panduan dosen |
|---|---|---|
| A | Matriks korelasi Pearson semua variabel numerik | Tahap 4a |
| B | Model regresi berganda + persamaan | 4b |
| C | R², Adjusted R², Uji-F, Uji-t per prediktor | 4c |
| D | Cek asumsi residual (normalitas & homoskedastisitas) | 4d |

> Scatter plot dengan garis tren dibuat oleh **Data Visualizer** (notebook 04). Kamu fokus ke angka & asumsi.
> ⚠️ Kalau data bersih resmi belum ada, kamu akan melihat peringatan DRAF SEMENTARA. Boleh latihan dulu; **jalankan ulang semua** setelah data resmi masuk.

In [ ]:
# ============ SETUP (jalankan sel ini PERTAMA) ============
# Satu-satunya yang boleh diubah: GITHUB_USER dan GITHUB_REPO (diisi sekali oleh Project Lead).
import os, sys, subprocess
GITHUB_USER = "Tur-ai"      # <- username GitHub pemilik repo
GITHUB_REPO = "udara-mobilitas-statsprob"     # <- nama repository
BRANCH = "main"

if "google.colab" in sys.modules:                      # kalau dijalankan di Google Colab
    PROYEK = "/content/proyek"
    if not os.path.exists(PROYEK):
        token = ""
        try:                                           # opsional: untuk repo PRIVATE (lihat docs/02_...)
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:
            pass
        url = f"https://{token}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, PROYEK],
                           capture_output=True, text=True)
        if r.returncode != 0:
            pesan = (r.stderr or "").replace(token.rstrip("@"), "***") if token else (r.stderr or "")
            raise RuntimeError("Gagal clone repo. Cek GITHUB_USER/GITHUB_REPO, atau lihat bagian "
                               "'Troubleshooting' di docs/02_panduan_github_colab.md\n" + pesan)
    else:
        subprocess.run(["git", "-C", PROYEK, "pull"], capture_output=True)   # ambil update terbaru
    os.chdir(PROYEK)
elif os.path.basename(os.getcwd()) == "notebooks":    # kalau dijalankan di Jupyter lokal
    os.chdir("..")
sys.path.insert(0, os.getcwd())



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Pastikan folder output tersedia
os.makedirs("../outputs/figures", exist_ok=True)
os.makedirs("../outputs/tables", exist_ok=True)

# Muat dataset bersih Kelompok 12
df = pd.read_csv("../data/Kelompok_12_data_bersih.csv")
print("Ukuran dataset:", df.shape)
print("Daftar kolom:", df.columns.tolist())
df.head()

## A. Matriks korelasi Pearson (Tahap 4a)

**Korelasi (r)** mengukur kekuatan & arah hubungan **linear** antara 2 variabel angka:
- r mendekati **+1** -> naik bersama (satu naik, yang lain naik)
- r mendekati **-1** -> berlawanan (satu naik, yang lain turun)
- r mendekati **0** -> tidak ada hubungan *linear*

Panduan kasar |r|: < 0.3 lemah, 0.3-0.7 sedang, > 0.7 kuat.
⚠️ **Korelasi bukan sebab-akibat.** Dua hal bisa bergerak bersamaan tanpa yang satu menyebabkan yang lain.

In [ ]:
# 1. Matriks Korelasi Pearson seluruh variabel numerik
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[num_cols].corr(method="pearson")
corr_matrix.to_csv("../outputs/tables/03_matriks_korelasi.csv")
print("=== Matriks Korelasi Pearson ===")
display(corr_matrix.round(4))

# Visualisasi Heatmap Korelasi
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Matriks Korelasi Pearson Variabel Numerik (Kelompok 12 - Tema B)")
plt.tight_layout()
plt.savefig("../outputs/figures/03_heatmap_korelasi.png", dpi=300)
plt.show()

# 2. Scatter Plot dengan Garis Tren: Prediktor Tema B vs Indeks_PM25
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Volume_Kendaraan vs Indeks_PM25
r1, p1 = stats.pearsonr(df["Volume_Kendaraan"], df["Indeks_PM25"])
sns.regplot(
    data=df, x="Volume_Kendaraan", y="Indeks_PM25",
    ax=axes[0], scatter_kws={"alpha": 0.6, "color": "steelblue"},
    line_kws={"color": "crimson", "linewidth": 2}
)
axes[0].set_title(f"Volume Kendaraan vs Indeks PM2.5\n(r = {r1:.3f}, p = {p1:.4e})")
axes[0].set_xlabel("Volume Kendaraan")
axes[0].set_ylabel("Indeks PM2.5")

# Plot 2: Kecepatan_Angin_KmJ vs Indeks_PM25
r2, p2 = stats.pearsonr(df["Kecepatan_Angin_KmJ"], df["Indeks_PM25"])
sns.regplot(
    data=df, x="Kecepatan_Angin_KmJ", y="Indeks_PM25",
    ax=axes[1], scatter_kws={"alpha": 0.6, "color": "seagreen"},
    line_kws={"color": "darkorange", "linewidth": 2}
)
axes[1].set_title(f"Kecepatan Angin (Km/Jam) vs Indeks PM2.5\n(r = {r2:.3f}, p = {p2:.4e})")
axes[1].set_xlabel("Kecepatan Angin (Km/Jam)")
axes[1].set_ylabel("Indeks PM2.5")

plt.tight_layout()
plt.savefig("../outputs/figures/03_scatter_tren_regresi.png", dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(korelasi, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True, linewidths=.5, ax=ax)
ax.set_title("Matriks korelasi Pearson")
plt.tight_layout()
simpan_gambar(fig, "04_heatmap_korelasi")
plt.show()

✏️ **ISI - Interpretasi:**
1. Variabel mana yang paling kuat berkorelasi dengan `Indeks_PM25`? Positif atau negatif? Masuk akal secara fisik? (mis. angin kencang menyebarkan polusi)
2. Apakah ada dua **prediktor** yang saling berkorelasi tinggi? (itu tanda **multikolinearitas** - dicek lagi lewat VIF di bawah)

_(tulis di sini)_

## B. Model Regresi Linear Berganda (Tahap 4b)

Regresi mencari garis (bidang) yang **paling dekat ke semua titik data**, dengan meminimalkan jumlah kuadrat selisih (residual). Metodenya **OLS** (Ordinary Least Squares).

- $y$ = **variabel respons** = `Indeks_PM25`
- $x_1$ = `Volume_Kendaraan`, $x_2$ = `Kecepatan_Angin_KmJ` = **prediktor**
- $b_0$ = intersep (nilai prediksi saat semua prediktor = 0)
- $b_1, b_2$ = **koefisien**: perubahan rata-rata PM2.5 jika prediktor itu naik 1 satuan, **sementara prediktor lain dianggap tetap**

In [ ]:
model = smf.ols("Indeks_PM25 ~ Volume_Kendaraan + Kecepatan_Angin_KmJ", data=df).fit()
print(model.summary())

### Cara membaca `summary` di atas
| Bagian | Artinya |
|---|---|
| `coef` | nilai b0, b1, b2 |
| `std err` | ketidakpastian koefisien |
| `t` dan `P>\|t\|` | **Uji-t parsial**: apakah prediktor itu berpengaruh nyata (p < 0.05 = ya) |
| `[0.025 0.975]` | CI 95% koefisien |
| `R-squared` | persentase variasi PM2.5 yang dijelaskan model |
| `Adj. R-squared` | R² yang "dihukum" kalau menambah prediktor tak berguna |
| `F-statistic`, `Prob (F-statistic)` | **Uji-F simultan**: apakah model secara keseluruhan berguna (p < 0.05 = ya) |

In [ ]:
# Definisikan variabel dependen (Y) dan independen (X)
target = "Indeks_PM25"
predictors = ["Volume_Kendaraan", "Kecepatan_Angin_KmJ"]

X = sm.add_constant(df[predictors])
y = df[target]

# Fit model Ordinary Least Squares (OLS)
model = sm.OLS(y, X).fit()
print(model.summary())

# Ekstrak parameter & persamaan regresi
b0 = model.params["const"]
b1 = model.params["Volume_Kendaraan"]
b2 = model.params["Kecepatan_Angin_KmJ"]

tanda_b1 = "+" if b1 >= 0 else "-"
tanda_b2 = "+" if b2 >= 0 else "-"
persamaan = f"Indeks_PM25_hat = {b0:.4f} {tanda_b1} {abs(b1):.4f}*(Volume_Kendaraan) {tanda_b2} {abs(b2):.4f}*(Kecepatan_Angin_KmJ)"

print("\n=== PERSAMAAN REGRESI LINEAR BERGANDA ===")
print(persamaan)

# Simpan ringkasan koefisien & statistik uji ke CSV
ringkasan_koef = pd.DataFrame({
    "Variabel": model.params.index,
    "Koefisien_b": model.params.values,
    "Std_Error": model.bse.values,
    "t_Statistik": model.tvalues.values,
    "p_value": model.pvalues.values,
    "CI_95_Bawah": model.conf_int()[0].values,
    "CI_95_Atas": model.conf_int()[1].values
})
ringkasan_koef.to_csv("../outputs/tables/03_hasil_regresi_ols.csv", index=False)

ringkasan_model = pd.DataFrame({
    "Metrik": ["R-squared", "Adjusted R-squared", "F-statistik", "Prob (F-statistik)", "Jumlah Observasi (n)"],
    "Nilai": [model.rsquared, model.rsquared_adj, model.fvalue, model.f_pvalue, int(model.nobs)]
})
ringkasan_model.to_csv("../outputs/tables/03_evaluasi_fit_model.csv", index=False)
display(ringkasan_koef)
display(ringkasan_model)

## C. Evaluasi model (Tahap 4c)

Jawab pertanyaan berikut dengan memakai angka di tabel atas:

1. **Uji-F simultan.** H0: $b_1 = b_2 = 0$ (model tidak berguna) vs H1: minimal satu $b \ne 0$. Lihat `p-value (Uji-F)`.
2. **Uji-t parsial** untuk tiap prediktor. H0: $b_i = 0$ vs H1: $b_i \ne 0$. Lihat kolom `p_value`.
3. **R² dan Adjusted R²**: berapa % variasi PM2.5 yang dijelaskan?  (R² = 0.40 berarti 40% naik-turunnya PM2.5 dijelaskan Volume & Angin; 60% sisanya oleh faktor lain.)

✏️ **ISI - Tabel evaluasi:**

| Komponen | Hasil | Keputusan / makna |
|---|---|---|
| Uji-F | F = ..., p = ... | Model (berguna / tidak) |
| Uji-t Volume | t = ..., p = ... | (berpengaruh nyata / tidak) |
| Uji-t Angin | t = ..., p = ... | (berpengaruh nyata / tidak) |
| R² / Adj R² | ... / ... | model menjelaskan ...% variasi |

✏️ **ISI - Makna koefisien dalam kalimat sehari-hari:**
_Contoh: "Setiap penambahan 100 kendaraan, rata-rata PM2.5 naik sekitar ... poin (dengan kecepatan angin dianggap tetap). Setiap kenaikan angin 1 km/jam, PM2.5 turun sekitar ... poin."_

In [ ]:
# What-if sederhana: dampak perubahan prediktor (dipakai Project Lead untuk rekomendasi)
b_vol, b_angin = model.params["Volume_Kendaraan"], model.params["Kecepatan_Angin_KmJ"]
print(f"Volume +100 kendaraan  -> PM2.5 berubah sekitar {100*b_vol:+.2f} poin")
print(f"Volume -100 kendaraan  -> PM2.5 berubah sekitar {-100*b_vol:+.2f} poin")
print(f"Angin  +5 km/jam       -> PM2.5 berubah sekitar {5*b_angin:+.2f} poin")

# Prediksi untuk skenario tertentu + selang prediksi 95%
skenario = pd.DataFrame({"Volume_Kendaraan": [300, 500, 700], "Kecepatan_Angin_KmJ": [10, 10, 10]})
pred = model.get_prediction(skenario).summary_frame(alpha=0.05)[["mean", "obs_ci_lower", "obs_ci_upper"]]
pred.columns = ["Prediksi_PM25", "Rentang95_Bawah", "Rentang95_Atas"]
pred = pd.concat([skenario, pred.round(1)], axis=1)
simpan_tabel(pred, "04_regresi_skenario_prediksi", index=False)
pred

## D. Pemeriksaan asumsi residual (Tahap 4d)

**Residual** = selisih nilai asli dengan prediksi model ($e = y - \hat{y}$). Regresi hanya bisa dipercaya jika residualnya "berperilaku baik":

| Asumsi | Artinya | Cara cek |
|---|---|---|
| **Normalitas residual** | residual berbentuk lonceng di sekitar 0 | Histogram, Q-Q plot, Shapiro-Wilk |
| **Homoskedastisitas** | sebaran residual **sama lebar** di semua nilai prediksi (tidak melebar seperti corong) | Plot *fitted vs residual*, uji Breusch-Pagan |
| **Linearitas** | hubungannya memang garis lurus | plot fitted vs residual: tidak boleh ada pola melengkung |
| **Tanpa multikolinearitas** | prediktor tidak saling terlalu mirip | VIF (< 5 aman, > 10 bermasalah) |

In [ ]:
residuals = model.resid
fitted = model.fittedvalues

# 1. Uji Formal Asumsi Residual
sw_stat, sw_p = stats.shapiro(residuals)
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(residuals, model.model.exog)

# 2. Cek Multikolinearitas (VIF)
vif_df = pd.DataFrame({
    "Variabel": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

print("=== UJI ASUMSI KLASIK REGRESI ===")
print(f"1. Normalitas Residual (Shapiro-Wilk) : W = {sw_stat:.4f}, p-value = {sw_p:.4f} -> {'Terpenuhi (Normal)' if sw_p > 0.05 else 'Tidak Normal (p <= 0.05)'}")
print(f"2. Homoskedastisitas (Breusch-Pagan)  : LM = {bp_lm:.4f}, p-value = {bp_lm_p:.4f} -> {'Terpenuhi (Homoskedastis)' if bp_lm_p > 0.05 else 'Heteroskedastisitas (p <= 0.05)'}")
print("3. Multikolinearitas (VIF < 10):")
display(vif_df[vif_df["Variabel"] != "const"])

# 3. Plot Diagnostik Residual (3 Panel)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# Panel A: Histogram Residual + Kurva Normal
sns.histplot(residuals, kde=True, stat="density", bins=20, ax=axes[0], color="mediumpurple", edgecolor="black")
grid_res = np.linspace(residuals.min(), residuals.max(), 200)
axes[0].plot(grid_res, stats.norm.pdf(grid_res, residuals.mean(), residuals.std(ddof=1)), "r--", lw=2, label="Kurva Normal")
axes[0].set_title(f"Histogram Residual\n(Shapiro p = {sw_p:.4f})")
axes[0].set_xlabel("Residual")
axes[0].legend()

# Panel B: Q-Q Plot Residual
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q Plot Residual")

# Panel C: Fitted Values vs Residuals (Homoskedastisitas)
axes[2].scatter(fitted, residuals, alpha=0.65, color="teal", edgecolors="k", linewidths=0.4)
axes[2].axhline(0, color="crimson", linestyle="--", linewidth=1.8)
axes[2].set_title(f"Fitted Values vs Residuals\n(Breusch-Pagan p = {bp_lm_p:.4f})")
axes[2].set_xlabel("Nilai Prediksi (Fitted Indeks_PM25)")
axes[2].set_ylabel("Residual")

plt.tight_layout()
plt.savefig("../outputs/figures/03_diagnostik_residual.png", dpi=300)
plt.show()

In [ ]:
# Uji formal
w, p_sh = stats.shapiro(resid)
bp_lm, bp_p, _, _ = het_breuschpagan(resid, model.model.exog)
X = model.model.exog
vif = pd.DataFrame({"Variabel": model.model.exog_names[1:],
                    "VIF": [variance_inflation_factor(X, i) for i in range(1, X.shape[1])]}).round(3)

print(f"Shapiro-Wilk residual : W = {w:.4f}, p = {p_sh:.4f} -> {'residual TIDAK normal' if p_sh < ALPHA else 'tidak ada bukti menyimpang dari normal'}")
print(f"Breusch-Pagan         : LM = {bp_lm:.3f}, p = {bp_p:.4f} -> {'ADA heteroskedastisitas' if bp_p < ALPHA else 'homoskedastisitas terpenuhi'}")
print("\nVIF (multikolinearitas):"); display(vif)

asumsi = pd.DataFrame([
    ["Normalitas residual (Shapiro-Wilk)", round(w, 4), round(p_sh, 4), "Terpenuhi" if p_sh >= ALPHA else "Dilanggar"],
    ["Homoskedastisitas (Breusch-Pagan)", round(bp_lm, 3), round(bp_p, 4), "Terpenuhi" if bp_p >= ALPHA else "Dilanggar"],
], columns=["Asumsi", "Statistik", "p_value", "Kesimpulan"])
simpan_tabel(asumsi, "04_uji_asumsi", index=False)
simpan_tabel(vif, "04_vif", index=False)
asumsi

✏️ **ISI - Kesimpulan asumsi (singkat):**
- Normalitas: _(histogram/Q-Q simetris? Shapiro?)_
- Homoskedastisitas: _(plot fitted vs residual: acak merata atau membentuk corong? Breusch-Pagan?)_
- Linearitas & multikolinearitas: _(ada pola melengkung? VIF berapa?)_
- **Dampaknya:** jika ada asumsi dilanggar, tulis jujur sebagai **keterbatasan** model dan hati-hati menafsirkan p-value.

💡 **Kalau homoskedastisitas dilanggar,** salah satu perbaikan sederhana adalah memakai *robust standard error* (sel opsional di bawah).

In [ ]:
# OPSIONAL: robust standard error (HC3). Dipakai jika Breusch-Pagan signifikan.
model_robust = smf.ols("Indeks_PM25 ~ Volume_Kendaraan + Kecepatan_Angin_KmJ", data=df).fit(cov_type="HC3")
print(model_robust.summary().tables[1])

## E. (Opsional / bonus) Model yang diperluas

Model utama (2 prediktor) adalah yang diminta dosen. Kalau tim ingin melihat apakah menambah variabel membuat model lebih baik, coba model 2 yang ditambah suhu, zona lokasi, jenis hari, dan jam sibuk. Variabel kategori otomatis diubah menjadi variabel dummy (0/1) oleh `C(...)`.

⚠️ **JANGAN memasukkan `Status_Udara` sebagai prediktor!** Status itu dihitung langsung dari PM2.5, jadi sama saja "menyontek jawaban" -> R² palsu tinggi.

In [ ]:
model2 = smf.ols("Indeks_PM25 ~ Volume_Kendaraan + Kecepatan_Angin_KmJ + Suhu_Celcius + C(Zona_Lokasi) + C(Jenis_Hari) + C(Jam_Sibuk)", data=df).fit()
print(model2.summary().tables[1])
bandingan = pd.DataFrame({
    "Model": ["Model 1 (Volume + Angin)", "Model 2 (diperluas)"],
    "R2": [model.rsquared, model2.rsquared], "Adj_R2": [model.rsquared_adj, model2.rsquared_adj],
    "AIC": [model.aic, model2.aic], "Jumlah_Prediktor": [2, len(model2.params) - 1]}).round(4)
simpan_tabel(bandingan, "04_perbandingan_model", index=False)
bandingan

✏️ **ISI:** apakah Adj R² naik banyak? Prediktor tambahan mana yang signifikan? (**AIC lebih kecil = model lebih baik**). Ini bisa jadi bahan rekomendasi: *faktor apa lagi yang perlu dipantau selain volume kendaraan?*
Ingat: **model utama untuk laporan tetap Model 1**; Model 2 hanya sebagai analisis tambahan.

## Serahkan hasil

In [ ]:
unduh_hasil("hasil_03_regression")

### ✅ Checklist selesai (Tahap 4)
- [ ] Matriks korelasi + p-value + heatmap + interpretasi
- [ ] Persamaan regresi tertulis (b0, b1, b2 ada angkanya)
- [ ] R², Adj R², Uji-F, Uji-t tiap prediktor + interpretasi dalam kalimat
- [ ] Histogram/Q-Q residual + fitted vs residual + Shapiro + Breusch-Pagan + VIF + kesimpulan
- [ ] Tabel what-if/skenario sudah dibagikan ke Project Lead (untuk rekomendasi)
- [ ] Hasil di-upload ke `outputs/`
- [ ] Aku bisa menjelaskan tanpa membaca: arti koefisien, bedanya R² & Adj R², kenapa residual dicek